# **<span style="color:blue">Notebook 5 - Prof. Dr. Humberto Elias Lopes - Teoria de Amostragem, Coleta de Dados e Modelagem de Probabilidade</span>**

## <span style="color:blue"> 1) Códigos para Amostragem Aleatória Simples </span>

**Exemplo 1**

O departamento de auditoria interna precisa verificar a regularidade de 12.000 lançamentos contábeis registrados no ERP em determinado mês. Utiliza-se um algoritmo de geração de números pseudoaleatórios em Python para selecionar, de forma independente e equiprovável, 400 faturas para conferência documental.

In [3]:
import pandas as pd
import numpy as np

# 1. Simulação do cadastro de faturas da empresa (População N = 12.000)
np.random.seed(42)
populacao_faturas = pd.DataFrame({
    'id_fatura': range(1, 12001),
    'valor_reais': np.random.uniform(150, 15000, size=12000).round(2),
})

# 2. Extração da Amostra Aleatória Simples (n = 400 faturas)
# replace=False garante amostragem sem reposição
amostra_auditoria = populacao_faturas.sample(n=400, replace=False, random_state=42) #random_state para reprodutibilidade

# 3. Visualização dos primeiros itens sorteados para checagem
print(f"Total de faturas na base (N): {len(populacao_faturas)}")
print(f"Total de faturas selecionadas (n): {len(amostra_auditoria)}\n")
print(amostra_auditoria)

Total de faturas na base (N): 12000
Total de faturas selecionadas (n): 400

      id_fatura  valor_reais
1935       1936     12104.09
6494       6495     14495.42
1720       1721      2262.40
9120       9121     13353.59
360         361      5914.32
...         ...          ...
9572       9573      1063.36
5020       5021     10053.87
668         669      2920.52
3742       3743     11392.16
47           48      7873.01

[400 rows x 2 columns]


**Exemplo 2**

Em um centro de distribuição logístico que despacha diariamente 2.500 pacotes numerados, o supervisor sorteia 100 pedidos via sorteio aleatório simples ao final do turno para inspeção dimensional e checagem de conformidade física do empacotamento.

In [ ]:
import pandas as pd
import numpy as np

# 1. População (N = 2.500 pacotes expedidos no dia)
np.random.seed(123)

dados_populacao = pd.DataFrame({
    'id_pacote': range(1001, 3501),
    'peso_kg': np.random.normal(loc=5.1, scale=0.8, size=2500).round(2) # loc = média, scale = desvio padrão

# 2. Extração da amostra aleatória simples (n = 100 pacotes)
# replace=False garante que o mesmo pacote não seja sorteado duas vezes
amostra_pacotes = dados_populacao.sample(n=100, replace=False, random_state=123)

# 3. Conferência rápida dos resultados
print(f"Total de pacotes na expedição (N): {len(dados_populacao)}")
print(f"Total de pacotes sorteados (n):    {len(amostra_pacotes)}")

# Exibe os 5 primeiros pacotes selecionados para a balança de conferência
print(amostra_pacotes.head())

## <span style="color:blue"> 2) Códigos para Amostragem Aleatória Estratificada </span>

**Exemplo 1**

Uma empresa de software por assinatura (SaaS) possui 1.000 clientes corporativos divididos em três faixas de faturamento: Pequeno, Médio e Grande. A diretoria precisa selecionar 100 clientes para avaliar a satisfação com o produto.

Se o sorteio fosse totalmente aleatório, haveria um grande risco de sortear pouquíssimas empresas do estrato "Grande", cuja perda financeira seria muito mais crítica. A amostragem estratificada proporcional assegura que cada porte tenha representação fiel ao seu peso na carteira.

In [4]:
import pandas as pd
import numpy as np

# 1. Criação de uma base simulada com 1.000 clientes
np.random.seed(42)

dados_clientes = pd.DataFrame({
    'id_cliente': range(1, 1001),
    'porte': np.random.choice(['Pequeno', 'Médio', 'Grande'], size=1000, p=[0.60, 0.30, 0.10])
})

# 2. Amostragem Estratificada Proporcional
# Retiramos 10% (frac=0.10) de cada estrato para compor uma amostra total de 100 empresas
amostra_clientes = (
    dados_clientes
    .groupby('porte', group_keys=False) # group_keys=False evita que o índice do grupo seja adicionado ao resultado
    .apply(lambda grupo: grupo.sample(frac=0.10, random_state=42)) # lambda é usado para aplicar a função sample em cada grupo
)

# 3. Conferência das proporções
print("Quantidade na população:")
print(dados_clientes['porte'].value_counts())

print("\nQuantidade na amostra (exatamente 10% de cada grupo):")
print(amostra_clientes['porte'].value_counts())

Quantidade na população:
porte
Pequeno    613
Médio      287
Grande     100
Name: count, dtype: int64

Quantidade na amostra (exatamente 10% de cada grupo):
porte
Pequeno    61
Médio      29
Grande     10
Name: count, dtype: int64


/var/folders/fz/9cbpzgyn6g3_1z_x9yqhf9580000gn/T/ipykernel_43882/3848208956.py:17: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda grupo: grupo.sample(frac=0.10, random_state=42)) # lambda é usado para aplicar a função sample em cada grupo


**Exemplo 2**

Uma rede varejista com 2.000 colaboradores, distribuídos entre Operação de Loja, Logística/Depósito e Escritório Corporativo, realizará uma pesquisa aprofundada sobre clima e liderança com 200 funcionários.

As rotinas e as percepções de trabalho nesses três setores são substancialmente diferentes. Para evitar que a opinião do escritório corporativo (menor contingente) seja ofuscada pelo chão de loja, a equipe de Recursos Humanos define a estratificação por departamento funcional.

In [ ]:
import pandas as pd
import numpy as np

# 1. Criação de uma base simulada com 2.000 colaboradores
np.random.seed(123)

dados_rh = pd.DataFrame({
    'matricula': range(1001, 3001),
    'departamento': np.random.choice(
        ['Loja', 'Logistica', 'Corporativo'], 
        size=2000, 
        p=[0.70, 0.20, 0.10]
    )
})

# 2. Amostragem Estratificada Proporcional
# Sorteamos 10% (frac=0.10) dos empregados de cada departamento
amostra_rh = (
    dados_rh
    .groupby('departamento', group_keys=False)
    .apply(lambda dept: dept.sample(frac=0.10, random_state=123))
)

# 3. Comparativo de representatividade
print("Distribuição percentual na população:")
print((dados_rh['departamento'].value_counts(normalize=True) * 100).round(1))

print("\nDistribuição percentual na amostra:")
print((amostra_rh['departamento'].value_counts(normalize=True) * 100).round(1))

## <span style="color:blue"> 3) Códigos para Amostragem por Conglomerados </span>

**Exemplo 1**

Uma rede de supermercados possui 40 lojas distribuídas pelo estado. A gerência de auditoria interna precisa avaliar se os preços praticados nas etiquetas físicas de gôndola conferem com os registros do sistema central.

Deslocar auditores até as 40 unidades acarretaria custo excessivo com transporte e diárias. Sabendo que cada loja comercializa um mix diversificado de produtos que reflete a operação geral da rede, a gerência decide sortear 4 lojas (conglomerados) e auditar todos os itens comercializados nessas 4 unidades.

In [ ]:
import pandas as pd
import numpy as np

# Amostragem por conglomerados em um estágio

# 1. Simulação da base com 40 lojas e 20 produtos por unidade
np.random.seed(42)

lojas = [f"Loja_{i:02d}" for i in range(1, 41)]
dados_lojas = []

for loja in lojas:
    for item in range(1, 21):  # 20 produtos monitorados por loja
        dados_lojas.append({
            'loja': loja,
            'id_produto': f"SKU_{item:03d}",
            'preco_gondola': round(np.random.uniform(10, 150), 2)  # Correção aplicada com round() nativo
        })

base_varejo = pd.DataFrame(dados_lojas)

# 2. Sorteio dos conglomerados (seleção probabilística de 4 lojas)
todas_as_lojas = pd.Series(base_varejo['loja'].unique()) # unique() retorna um array, então convertemos para Series para usar sample()
lojas_sorteadas = todas_as_lojas.sample(n=4, random_state=42)

# 3. Extração integral de todos os itens das unidades sorteadas
amostra_conglomerados = base_varejo[base_varejo['loja'].isin(lojas_sorteadas)]

# 4. Painel de verificação didática
print("AUDITORIA POR CONGLOMERADOS: REDE DE SUPERMERCADOS")
print("=" * 55)
print(f"Total de lojas na rede (População de Conglomerados): {len(todas_as_lojas)}")
print(f"Lojas selecionadas no sorteio:                     {list(lojas_sorteadas.values)}")
print(f"Total de produtos auditados na amostra:            {len(amostra_conglomerados)}")
print(f"Preço médio apurado na amostra:                    R$ {amostra_conglomerados['preco_gondola'].mean():.2f}")
print("-" * 55)
print("Primeiros registros coletados para fiscalização em campo:")
print(amostra_conglomerados.head())

**Exemplo 2**

Uma transportadora urbana opera em 20 microrregiões logísticas (conglomerados) de uma região metropolitana. Cada microrregião atende a bairros com residências, comércios e pequenas indústrias, apresentando perfil misto de entrega.

Para mensurar o tempo médio de parada por entrega e recalcular o planejamento das rotas, a gerência de logística instala rastreadores avançados nos veículos que cobrem 3 microrregiões sorteadas, coletando todas as entregas realizadas dentro dessas áreas durante uma semana de trabalho.

In [ ]:
import pandas as pd
import numpy as np

# Amostragem por conglomerados em um estágio

# 1. Simulação da base com 20 microrregiões e 15 entregas por região
np.random.seed(101)

regioes = [f"Regiao_{r:02d}" for r in range(1, 21)]
dados_entregas = []

for regiao in regioes:
    for entrega in range(1, 16):  # 15 entregas por microrregião
        dados_entregas.append({
            'microrregiao': regiao,
            'id_entrega': f"ENT_{entrega:03d}",
            # Correção: uso da função embutida round() do Python para escalares
            'tempo_parada_min': round(np.random.normal(loc=18.0, scale=4.0), 1)
        })

base_logistica = pd.DataFrame(dados_entregas)

# 2. Sorteio dos conglomerados (seleção probabilística de 3 microrregiões)
todas_as_regioes = pd.Series(base_logistica['microrregiao'].unique())
regioes_sorteadas = todas_as_regioes.sample(n=3, random_state=101)

# 3. Extração integral de todas as entregas das microrregiões sorteadas
amostra_conglomerado = base_logistica[base_logistica['microrregiao'].isin(regioes_sorteadas)]

# 4. Painel de verificação e interpretação gerencial
print("AUDITORIA LOGÍSTICA: AMOSTRAGEM POR CONGLOMERADOS")
print("=" * 60)
print(f"Total de microrregiões atendidas (População N): {len(todas_as_regioes)}")
print(f"Microrregiões sorteadas:                      {list(regioes_sorteadas.values)}")
print(f"Total de entregas auditadas na amostra (n):    {len(amostra_conglomerado)}")
print(f"Tempo médio de parada observado:              {amostra_conglomerado['tempo_parada_min'].mean():.2f} minutos")
print("-" * 60)
print("Primeiros registros coletados para análise operacional:")
print(amostra_conglomerado.head())

**Exemplo 3**

Uma rede varejista de vestuário e calçados opera $N = 50$ lojas físicas distribuídas em diferentes shopping centers e centros comerciais de um estado. A diretoria de operações suspeita de divergências entre o estoque físico registrado em gôndola e os saldos reportados no sistema ERP (perdas, furtos ou falhas de inventário).

Auditar todas as 50 lojas implicaria custos elevados de deslocamento e equipes dedicadas. Por outro lado, auditar todos os itens de apenas 2 ou 3 lojas (estágio único) consumiria tempo excessivo em poucas localidades, sem cobrir a heterogeneidade operacional da rede.

Decisão gerencial:

Adotar a amostragem por conglomerados em dois estágios:

1) Primeiro estágio (macro - conglomerados): selecionar aleatoriamente $m = 5$ lojas físicas entre as 50 disponíveis (sorteio de conglomerados primários).

2) Segundo estágio (micro - unidades elementares): dentro de cada uma das 5 lojas selecionadas, realizar uma amostragem aleatória simples (AAS) de $k = 30$ itens (SKUs), totalizando uma amostra de $n = 150$ produtos para contagem física imediata.

In [ ]:
import numpy as np
import pandas as pd

# Amostragem por conglomerados em dois estágios

# ------------------------------------------------------------------------------
# 1. GERAÇÃO DA BASE POPULACIONAL (50 LOJAS COM 200 ITENS CADA = 10.000 REGISTROS)
# ------------------------------------------------------------------------------
np.random.seed(42)

total_lojas = 50
itens_por_loja = 200
total_populacao_N = total_lojas * itens_por_loja

lista_lojas = [f"Loja_{i:02d}" for i in range(1, total_lojas + 1)]

dados_populacao = pd.DataFrame({
    'id_loja': np.repeat(lista_lojas, itens_por_loja),
    'codigo_sku': [f"SKU_{j:04d}" for j in range(1, itens_por_loja + 1)] * total_lojas,
    'divergencia_inventario_reais': np.random.exponential(scale=18.5, size=total_populacao_N).round(2)
})

print(f"Total de lojas cadastradas (Conglomerados): {dados_populacao['id_loja'].nunique()}")
print(f"Total de itens no cadastro geral (População N): {len(dados_populacao)} produtos\n")

# ------------------------------------------------------------------------------
# 2. ESTÁGIO 1: SORTEIO ALEATÓRIO DOS CONGLOMERADOS (LOJAS)
# ------------------------------------------------------------------------------
# Seleção de m = 5 lojas sem reposição
m_lojas_sorteadas = 5

conglomerados_disponiveis = pd.Series(dados_populacao['id_loja'].unique())
lojas_selecionadas = conglomerados_disponiveis.sample(
    n=m_lojas_sorteadas, 
    replace=False, 
    random_state=101
).values

print("=" * 60)
print("ESTÁGIO 1: CONGLOMERADOS SELECIONADOS")
print("=" * 60)
print(f"Lojas que receberão as equipes de auditoria: {list(lojas_selecionadas)}")

# Filtragem dos dados apenas para as lojas selecionadas no Estágio 1
base_primeiro_estagio = dados_populacao[dados_populacao['id_loja'].isin(lojas_selecionadas)]

# ------------------------------------------------------------------------------
# 3. ESTÁGIO 2: AMOSTRAGEM ALEATÓRIA SIMPLES DENTRO DE CADA LOJA SORTEADA
# ------------------------------------------------------------------------------
# Dentro de cada conglomerado selecionado, sorteiam-se k = 30 SKUs
k_itens_por_loja = 30

amostra_final = (
    base_primeiro_estagio
    .groupby('id_loja', group_keys=False)
    .apply(lambda grupo: grupo.sample(n=k_itens_por_loja, replace=False, random_state=42))
    .reset_index(drop=True)
)

# ------------------------------------------------------------------------------
# 4. PAINEL DE CONFERÊNCIA GERENCIAL E ESTATÍSTICA
# ------------------------------------------------------------------------------
print("\n" + "=" * 60)
print("ESTÁGIO 2: RESUMO DA AMOSTRA FINAL COLETADA")
print("=" * 60)
print("Itens auditados por conglomerado:")
print(amostra_final['id_loja'].value_counts())

print("\nMétricas da Amostra Final:")
print(f"Tamanho total da amostra (n): {len(amostra_final)} itens auditados")
print(f"Divergência média amostral por item: R$ {amostra_final['divergencia_inventario_reais'].mean():.2f}")
print(f"Desvio padrão amostral: R$ {amostra_final['divergencia_inventario_reais'].std(ddof=1):.2f}")

print("\nPrimeiros 5 registros da amostra de auditoria:")
print(amostra_final.head())

## <span style="color:blue"> 4) Códigos para Amostragem Aleatória Sistemática </span>

**Exemplo 1**

Uma linha automatizada de envase de sucos produz $N = 2.000$ garrafas por turno operacional. O gerente de qualidade precisa auditar $n = 100$ garrafas ao longo da jornada para checar o volume real de envase (alvo: 500 ml) e monitorar a dispersão do maquinário. Como as garrafas passam sequencialmente pela esteira, o sorteio aleatório tradicional causaria atrito logístico. 

A equipe adota a amostragem sistemática com intervalo $k = 2000 / 100 = 20$. Sorteia-se uma garrafa inicial entre a 1ª e a 20ª, recolhendo subsequentemente a cada 20 unidades que cruzam o sensor de pesagem.

In [7]:
import numpy as np
import pandas as pd

# -------------------------------------------------------------
# 1. SIMULAÇÃO DA LINHA DE ENVASE (POPULAÇÃO N = 2.000 GARRAFAS)
# -------------------------------------------------------------
np.random.seed(42)

tamanho_populacao_N = 2000
tamanho_amostra_n = 100

dados_envase = pd.DataFrame({
    'ordem_esteira': range(1, tamanho_populacao_N + 1),
    'volume_ml': np.random.normal(loc=501.2, scale=2.5, size=tamanho_populacao_N).round(2)
})

# -------------------------------------------------------------
# 2. PLANO AMOSTRAL: AMOSTRAGEM SISTEMÁTICA
# -------------------------------------------------------------
# Passo A: Determinação do salto intervalar (k)
intervalo_k = tamanho_populacao_N // tamanho_amostra_n  # k = 20

# Passo B: Ponto de partida aleatório entre 1 e k
np.random.seed(10)  # Semente para reprodutibilidade didática
ponto_partida_r = np.random.randint(1, intervalo_k + 1)

# Passo C: Construção dos índices de seleção sistemática
indices_sistematicos = range(ponto_partida_r - 1, tamanho_populacao_N, intervalo_k)

# Passo D: Extração das garrafas selecionadas
amostra_qualidade = dados_envase.iloc[indices_sistematicos].reset_index(drop=True)

# -------------------------------------------------------------
# 3. INTERPRETAÇÃO E CONFERÊNCIA GERENCIAL
# -------------------------------------------------------------
print("AUDITORIA DE ENVASE: CONTROLE ESTATÍSTICO DE QUALIDADE")
print("=" * 60)
print(f"Total produzido no turno (N):      {tamanho_populacao_N} garrafas")
print(f"Tamanho da amostra necessária (n): {tamanho_amostra_n} garrafas")
print(f"Intervalo de salto (k):            A cada {intervalo_k} garrafas")
print(f"Ponto de partida sorteado (r):     Garrafa nº {ponto_partida_r}")
print(f"Total de itens recolhidos:         {len(amostra_qualidade)} garrafas")
print(f"Volume médio apurado:              {amostra_qualidade['volume_ml'].mean():.2f} ml")
print("-" * 60)
print("Primeiras 5 garrafas coletadas na esteira:")
print(amostra_qualidade.head())

AUDITORIA DE ENVASE: CONTROLE ESTATÍSTICO DE QUALIDADE
Total produzido no turno (N):      2000 garrafas
Tamanho da amostra necessária (n): 100 garrafas
Intervalo de salto (k):            A cada 20 garrafas
Ponto de partida sorteado (r):     Garrafa nº 10
Total de itens recolhidos:         100 garrafas
Volume médio apurado:              501.61 ml
------------------------------------------------------------
Primeiras 5 garrafas coletadas na esteira:
   ordem_esteira  volume_ml
0             10     502.56
1             30     500.47
2             50     496.79
3             70     499.59
4             90     502.48


**Exemplo 2**

O departamento de compliance financeiro de um marketplace processou $N = 1.200$ transferências via Pix durante um final de semana promocional. Para verificar se os procedimentos antifraude e retenção tributária operaram conforme as diretrizes legais, o auditor sênior determina a inspeção documental de $n = 60$ transações.

A listagem é extraída ordenada cronologicamente por horário de autorização bancária. Com $k = 1200 / 60 = 20$, sorteia-se uma transação inicial entre as 20 primeiras e extrai-se a cada vigésimo pagamento registrado.

In [8]:
import numpy as np
import pandas as pd

# -------------------------------------------------------------
# 1. SIMULAÇÃO DO LIVRO DE TRANSAÇÕES (POPULAÇÃO N = 1.200)
# -------------------------------------------------------------
np.random.seed(123)

tamanho_populacao_N = 1200
tamanho_amostra_n = 60

dados_transacoes = pd.DataFrame({
    'id_transacao': [f"PIX_{i:04d}" for i in range(1, tamanho_populacao_N + 1)],
    'valor_transacao': np.random.exponential(scale=280.0, size=tamanho_populacao_N).round(2),
    'status_conformidade': np.random.choice(['Regular', 'Requer Ajuste'], size=tamanho_populacao_N, p=[0.97, 0.03])
})

# -------------------------------------------------------------
# 2. PLANO AMOSTRAL: AMOSTRAGEM SISTEMÁTICA
# -------------------------------------------------------------
intervalo_k = tamanho_populacao_N // tamanho_amostra_n  # k = 20

# Sorteio do primeiro registro a ser auditado
np.random.seed(99)
ponto_partida_r = np.random.randint(1, intervalo_k + 1)

# Geração das posições regulares com a função range nativa do Python
posicoes_selecionadas = list(range(ponto_partida_r - 1, tamanho_populacao_N, intervalo_k))

# Filtragem posicional via .iloc[]
amostra_auditoria = dados_transacoes.iloc[posicoes_selecionadas].reset_index(drop=True)

# -------------------------------------------------------------
# 3. RELATÓRIO EXECUTIVO PARA CONTROLES INTERNOS
# -------------------------------------------------------------
print("AUDITORIA DE TRANSAÇÕES FINANCEIRAS: AMOSTRAGEM SISTEMÁTICA")
print("=" * 65)
print(f"Universo de transações (N):        {tamanho_populacao_N}")
print(f"Meta de auditoria (n):             {tamanho_amostra_n}")
print(f"Salto intervalar (k):              {intervalo_k}")
print(f"Primeira transação auditada:       Posição {ponto_partida_r} ({dados_transacoes.iloc[ponto_partida_r - 1]['id_transacao']})")
print(f"Volume total auditado:             {len(amostra_auditoria)} operações")
print(f"Ticket médio na amostra:           R$ {amostra_auditoria['valor_transacao'].mean():.2f}")
print(f"Inconsistências encontradas:       {(amostra_auditoria['status_conformidade'] == 'Requer Ajuste').sum()}")
print("-" * 65)
print("Primeiros registros selecionados para análise:")
print(amostra_auditoria.head())

AUDITORIA DE TRANSAÇÕES FINANCEIRAS: AMOSTRAGEM SISTEMÁTICA
Universo de transações (N):        1200
Meta de auditoria (n):             60
Salto intervalar (k):              20
Primeira transação auditada:       Posição 2 (PIX_0002)
Volume total auditado:             60 operações
Ticket médio na amostra:           R$ 294.54
Inconsistências encontradas:       2
-----------------------------------------------------------------
Primeiros registros selecionados para análise:
  id_transacao  valor_transacao status_conformidade
0     PIX_0002            94.38             Regular
1     PIX_0022           530.13             Regular
2     PIX_0042            34.40             Regular
3     PIX_0062           246.89             Regular
4     PIX_0082           329.72             Regular


## <span style="color:blue"> 5) Códigos para Amostragem por Conveniência </span>

**Exemplo 1**

A equipe de produto de um comércio eletrônico precisa identificar travamentos imediatos na etapa de finalização de compra antes do lançamento oficial. O coordenador aborda os 5 primeiros analistas de outros departamentos que ingressam no refeitório corporativo.

In [ ]:
# 1. Base simulada com a ordem de chegada de colaboradores ao refeitório
dados_chegada = pd.DataFrame({
    'matricula': [101, 102, 103, 104, 105, 106, 107, 108],
    'departamento': ['RH', 'Fiscal', 'Compras', 'Vendas', 'Fiscal', 'Juridico', 'TI', 'Logistica']
})

# 2. Amostragem por Conveniência: selecionam-se os primeiros elementos disponíveis
amostra_conveniencia_1 = dados_chegada.head(4)

print("Amostra por Conveniência (Primeiros a chegar):")
print(amostra_conveniencia_1)

**Exemplo 2**

O gerente de uma agência bancária física deseja entender as queixas sobre o sistema de ar-condicionado. Em vez de uma amostragem probabilística de todos os correntistas cadastrados, ele entrega um questionário impresso apenas aos clientes sentados na fila de espera próxima à sua mesa em determinado momento do dia.

In [ ]:
# 1. Registro dos clientes presentes no salão de atendimento
clientes_espera = pd.DataFrame({
    'id_cliente': [201, 202, 203, 204, 205],
    'tempo_espera_min': [12, 5, 22, 18, 9],
    'posicao_fila': ['Fila Prioritaria', 'Mesa Gerente', 'Mesa Gerente', 'Caixa Rapido', 'Mesa Gerente']
})

# 2. Coleta conveniente: selecionam-se apenas os clientes no campo visual do gerente
amostra_conveniencia_2 = clientes_espera[clientes_espera['posicao_fila'] == 'Mesa Gerente']

print("Amostra por Conveniência (Clientes ao lado da gerência):")
print(amostra_conveniencia_2)

## <span style="color:blue"> 6) Códigos para Amostragem Intencional (Por Julgamento) </span>

**Exemplo 1**

O comitê de crédito de uma instituição financeira possui centenas de operações com atraso, mas decide analisar pormenorizadamente apenas os contratos com saldo devedor superior a R$ 100.000,00 e atraso superior a 60 dias, julgando serem os de maior risco sistêmico.

In [ ]:
# 1. Base cadastral de contratos inadimplentes
carteira_inadimplente = pd.DataFrame({
    'contrato': [1001, 1002, 1003, 1004, 1005],
    'saldo_devedor': [25000, 140000, 12000, 310000, 85000],
    'dias_atraso': [15, 90, 45, 120, 70]
})

# 2. Critério de julgamento do auditor: Alto valor (> 100 mil) E Atraso prolongado (> 60 dias)
regra_risco = (carteira_inadimplente['saldo_devedor'] > 100000) & (carteira_inadimplente['dias_atraso'] > 60)
amostra_julgamento_1 = carteira_inadimplente[regra_risco]

print("Amostra por Julgamento (Contratos de Alto Risco):")
print(amostra_julgamento_1)

**Exemplo 2**

Uma desenvolvedora de software corporativo busca feedback qualitativo sobre uma nova funcionalidade. Em vez de sortear clientes aleatórios, o gerente de produto seleciona expressamente os clientes categorizados internamente como "Usuários Avançados" (Power Users).

In [ ]:
# 1. Base de usuários cadastrados
usuarios_software = pd.DataFrame({
    'empresa': ['Alpha Ltda', 'Beta S/A', 'Gama Tech', 'Delta Servicos', 'Epsilon ME'],
    'perfil_uso': ['Iniciante', 'Power User', 'Intermediario', 'Power User', 'Iniciante']
})

# 2. Amostragem intencional focada no perfil que domina a ferramenta
amostra_julgamento_2 = usuarios_software[usuarios_software['perfil_uso'] == 'Power User']

print("Amostra por Julgamento (Apenas 'Power Users'):")
print(amostra_julgamento_2)

## <span style="color:blue"> 7) Códigos para Amostragem por Cotas </span>

**Exemplo 1**

A área de trade marketing de um shopping center precisa colher a opinião de 4 clientes, respeitando a meta de 2 jovens (até 30 anos) e 2 seniores (acima de 30 anos), entrevistando os primeiros que aceitarem responder na praça de alimentação.

In [ ]:
# 1. Fila de transeuntes abordados pelos entrevistadores
transeuntes = pd.DataFrame({
    'id': [1, 2, 3, 4, 5, 6, 7],
    'faixa_etaria': ['Jovem', 'Jovem', 'Senior', 'Jovem', 'Senior', 'Senior', 'Jovem']
})

# 2. Definição da cota: capturar os primeiros 2 registros de cada faixa
cota_por_grupo = 2
amostra_cotas_1 = transeuntes.groupby('faixa_etaria').head(cota_por_grupo)

print("Amostra por Cotas (2 Jovens e 2 Seniores capturados):")
print(amostra_cotas_1)

**Exemplo 2**

O departamento de crédito de uma cooperativa quer colher impressões de clientes sobre uma nova linha de financiamento, exigindo uma cota exata de 3 clientes da Agência Física e 3 clientes do Canal Digital, colhendo os primeiros que responderem ao chamado.

In [ ]:
# 1. Registro de solicitações que deram entrada na central
solicitacoes = pd.DataFrame({
    'protocolo': [501, 502, 503, 504, 505, 506, 507, 508],
    'canal': ['Digital', 'Fisico', 'Digital', 'Digital', 'Fisico', 'Fisico', 'Fisico', 'Digital']
})

# 2. Extração da cota: exatamente 3 observações de cada canal
cota_canal = 3
amostra_cotas_2 = solicitacoes.groupby('canal').head(cota_canal)

print("Amostra por Cotas (3 de cada canal):")
print(amostra_cotas_2)

## <span style="color:blue"> 8) Códigos para Amostragem por Bola de Neve (Snowball) </span>

**Exemplo 1**

Um pesquisador corporativo estuda práticas de gestão de crise em empresas recuperadas judicialmente. Como se trata de um público reservado, ele inicia com 1 diretor conhecido, que indica 2 colegas do setor, os quais por sua vez indicam novos participantes.

In [ ]:
# 1. Rede de contatos profissionais mapeada em um dicionario
rede_indicacoes = {
    'Diretor_Inicial_A': ['Diretor_B', 'Diretor_C'],
    'Diretor_B': ['Diretor_D'],
    'Diretor_C': ['Diretor_E', 'Diretor_F']
}

# 2. Cadeia de amostragem Bola de Neve (início com a "semente")
amostra_bola_de_neve_1 = ['Diretor_Inicial_A']

# Primeira rodada de indicações a partir do contato inicial
amostra_bola_de_neve_1.extend(rede_indicacoes['Diretor_Inicial_A'])

# Segunda rodada: adicionar contatos indicados pelo Diretor_B
amostra_bola_de_neve_1.extend(rede_indicacoes['Diretor_B'])

# Conversão para DataFrame didático
df_amostra_neve = pd.DataFrame({'participantes_recrutados': amostra_bola_de_neve_1})

print("Amostra Bola de Neve (Cadeia de Diretores):")
print(df_amostra_neve)

**Exemplo 2**

A área de compras e inovação aberta de uma holding industrial precisa prospectar fornecedores nacionais com patentes em computação quântica aplicada à logística. Sem sindicato ou listagem oficial estruturada, o comprador consulta um laboratório parceiro, que indica as empresas pioneiras com as quais colabora.

In [ ]:
# 1. Lista estruturada de indicações sucessivas
indicacoes_parceiros = [
    {'origem': 'Laboratorio_Universidade', 'empresa_indicada': 'QuantumLog'},
    {'origem': 'QuantumLog', 'empresa_indicada': 'OptimaBits'},
    {'origem': 'OptimaBits', 'empresa_indicada': 'NanoRoute'}
]

# 2. Consolidação da amostra obtida pela rede de referências
amostra_bola_de_neve_2 = pd.DataFrame(indicacoes_parceiros)

print("Amostra Bola de Neve (Cadeia de Prospecção Tecnológica):")
print(amostra_bola_de_neve_2[['empresa_indicada']])

## <span style="color:blue"> 9) Matriz Comparativa dos Métodos de Amostragem </span>

| Método | Classificação | Critério Central de Seleção | Vantagem Principal em Negócios | Limitação Primária | Contexto Típico de Aplicação |
| :--- | :--- | :--- | :--- | :--- | :--- |
| **Aleatória Simples (AAS)** | Probabilística | Sorteio direto equiprovável ($p = n/N$) | Ausência de viés de seleção; inferência estatística pura | Exige lista exaustiva e atualizada da população (*sampling frame*) | Auditorias contábeis, conferência fiscal de notas e testes laboratoriais de controle de qualidade |
| **Estratificada** | Probabilística | Divisão por subgrupos homogêneos e sorteio interno | Redução do erro padrão; garantia de representatividade de nichos vitais | Exige conhecimento prévio das variáveis de segmentação | Pesquisas de satisfação de clientes segmentados por faturamento (ex.: PME vs. Enterprise) |
| **Por Conglomerados** | Probabilística | Sorteio de blocos heterogêneos (*clusters*) | Redução substancial de custos de deslocamento e logística | Maior erro amostral se os conglomerados forem internamente homogêneos | Pesquisas de mercado presenciais em redes de varejo, filiais físicas ou setores censitários urbanos |
| **Sistemática** | Probabilística | Salto intervalar constante ($k \approx N/n$) | Facilidade operacional em fluxos contínuos sem cadastro prévio | Risco severo de viés diante de periodicidades ou ciclos ocultos | Controle estatístico de processos em linhas de envase industrial e fluxos contínuos de caixas |
| **Por Conveniência** | Não Probabilística | Facilidade de acesso imediato e disponibilidade | Menor custo orçamentário e execução rápida | Alto viés de autosseleção; impede generalizações populacionais | Pré-testes de questionários de pesquisa, validação preliminar de interfaces digitais (UX) |
| **Intencional (Por Julgamento)** | Não Probabilística | Escolha deliberada por critério técnico de especialista | Foco direcionado a unidades de alto risco, valor ou densidade estratégica | Subjetividade elevada; reflete preconcepções do decisor | Auditoria investigativa de fraudes de grande vulto e painéis com lideranças executivas |
| **Por Cotas** | Não Probabilística | Preenchimento de metas por perfil demográfico sem sorteio | Assegura diversidade estrutural mínima na coleta de campo | Viés do entrevistador na abordagem final das pessoas que compõem a cota | Sondagens rápidas de opinião em centros comerciais e pesquisas de consumo em praças de alimentação |
| **Bola de Neve (Snowball)** | Não Probabilística | Cadeia de indicações sucessivas a partir de contatos-chave | Viabiliza estudos de públicos restritos ou de difícil localização | Homofilia; a amostra tende a refletir a rede fechada dos respondentes iniciais | Entrevistas sobre governança em empresas familiares, fundadores de *startups deep tech* e investidores anjo |